In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [6]:
# ================================
# 🚀 Optimized GraphCodeBERT – Vulnerability Detection (Fully Fixed)
# ================================

!pip install transformers datasets accelerate -q

import os, random, numpy as np, pandas as pd, torch
from datasets import Dataset, ClassLabel
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, set_seed
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ---------- 1. Reproducibility ----------
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---------- 2. Find and load CSV ----------
file_path = None
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".csv"):
            file_path = os.path.join(root, f)
            break
if not file_path:
    raise FileNotFoundError("No CSV file found in /kaggle/input")

df = pd.read_csv(file_path)
print("Original columns:", df.columns.tolist())

# --- Auto-rename columns ---
col_mapping = {
    "function": "code", "func": "code", "code_snippet": "code",
    "target": "label", "vul": "label"
}
df.rename(columns={k: v for k, v in col_mapping.items() if k in df.columns}, inplace=True)
df = df[["code", "label"]].dropna()
df["label"] = df["label"].astype(int)

print(f"Dataset size: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}")

# ---------- 3. Compute class weights ----------
labels = df["label"].values
class_counts = np.bincount(labels)
class_weights = torch.tensor(
    [len(labels) / (2.0 * cnt) if cnt > 0 else 1.0 for cnt in class_counts],
    dtype=torch.float32
)
print("Class weights:", class_weights.tolist())

# ---------- 4. Convert to Dataset and cast label to ClassLabel ----------
dataset = Dataset.from_pandas(df)
num_classes = df["label"].nunique()
dataset = dataset.cast_column("label", ClassLabel(num_classes=num_classes))

# Stratified split
split = dataset.train_test_split(test_size=0.1, seed=SEED, stratify_by_column="label")
train_ds, eval_ds = split["train"], split["test"]

# ---------- 5. Model & tokenizer ----------
model_name = "microsoft/graphcodebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_classes)

# ---------- 6. Tokenization ----------
def tokenize_fn(examples):
    return tokenizer(
        examples["code"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_train = train_ds.map(tokenize_fn, batched=True, remove_columns=["code"])
tokenized_eval  = eval_ds.map(tokenize_fn, batched=True, remove_columns=["code"])

# ---------- 7. Custom Trainer with weighted loss ----------
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ---------- 8. Metrics ----------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "f1": f1, "precision": prec, "recall": rec}

# ---------- 9. Training arguments (FIXED: eval_strategy) ----------
training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
    disable_tqdm=False
)

# ---------- 10. Trainer (tokenizer removed) ----------
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

# ---------- 11. Train & evaluate ----------
trainer.train()
results = trainer.evaluate()
print("\n📊 Final Evaluation:", {k: round(v, 4) for k, v in results.items()})

# ---------- 12. Save ----------
trainer.save_model("/kaggle/working/graphcodebert-primevul")
tokenizer.save_pretrained("/kaggle/working/graphcodebert-primevul")
print("✅ Model saved")

Original columns: ['code', 'label']
Dataset size: 2732
Label distribution:
label
0    1545
1    1187
Name: count, dtype: int64
Class weights: [0.8841423988342285, 1.150800347328186]


Casting the dataset:   0%|          | 0/2732 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/2458 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


📊 Final Evaluation: {'eval_loss': 0.6747, 'eval_accuracy': 0.5876, 'eval_f1': 0.5498, 'eval_precision': 0.5227, 'eval_recall': 0.5798, 'eval_runtime': 6.7504, 'eval_samples_per_second': 40.59, 'eval_steps_per_second': 10.222, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved
